### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="pancreatic_cancer_mouse_detection",
    dataset_year="2003",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="Other",
    # We get it from the following website, as we cannot find the original source.
    original_dataset_source_download_link="https://home.ccr.cancer.gov/ncifdaproteomics/ppatterns.asp",
    download_description=r"""
We download the data to a predefined folder.

mkdir -p local-data-warehouse/pancreatic_cancer_mouse_detection && \
wget -P local-data-warehouse/pancreatic_cancer_mouse_detection https://home.ccr.cancer.gov/ncifdaproteomics/CancerCellPanINDataBinned.zip && \
unzip local-data-warehouse/pancreatic_cancer_mouse_detection/CancerCellPanINDataBinned.zip -d local-data-warehouse/pancreatic_cancer_mouse_detection/ && \
rm local-data-warehouse/pancreatic_cancer_mouse_detection/CancerCellPanINDataBinned.zip && \
mv local-data-warehouse/pancreatic_cancer_mouse_detection/CancerCellPanINDataBinned/02-Control/PanINCTL local-data-warehouse/pancreatic_cancer_mouse_detection/ && \
mv local-data-warehouse/pancreatic_cancer_mouse_detection/CancerCellPanINDataBinned/03-PanIN local-data-warehouse/pancreatic_cancer_mouse_detection && \
rm -r local-data-warehouse/pancreatic_cancer_mouse_detection/CancerCellPanINDataBinned
""",
    # References
    academic_reference_bibtex=r""""@article{hingorani2003preinvasive,
  title={Preinvasive and invasive ductal pancreatic cancer and its early detection in the mouse},
  author={Hingorani, Sunil R and Petricoin, Emanuel F and Maitra, Anirban and Rajapakse, Vinodh and King, Catrina and Jacobetz, Michael A and Ross, Sally and Conrads, Thomas P and Veenstra, Timothy D and Hitt, Ben A and others},
  journal={Cancer cell},
  volume={4},
  number={6},
  pages={437--450},
  year={2003},
  publisher={Elsevier}
}"
""",
    academic_reference_bibtex_key="hingorani2003preinvasive",
    license="None", # Not given, likely public
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
The task in the original data is grouped as multiple serums per mouse were taken.
- We merge all samples located in the .csv files into one DataFrame and recover target labels, group IDs, and feature names.
- We label the samples based on name of the folder they are located in: "PanINCTL" translates to healthy animals cases while "03-PanIN" to individuals with a pancreatic cancer.
- Anomaly: The data from the website has one more mouse than accounted for in the paper (73 vs (33 + 39))) but the same number of serums (181).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="HasCancer",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="HasCancer",
    group_on="mouse_id",
    group_labels="per_group",
)

## Preprocessing

In [ ]:
import pandas as pd

BASE_DIR = dataset_mold.path
LABEL_MAP = {
  "PanINCTL": "No",
  "03-PanIN": "Yes",
}

rows = []
for folder, label in LABEL_MAP.items():
  for filepath in sorted((BASE_DIR / folder).glob("*.csv")):
      sample = pd.read_csv(filepath)

      # Filename format: {id}_{group_id}_{rest}.txt_PQ.csv
      group_id = filepath.stem.split("_")[1]

      row = dict(zip("M/Z:" + sample["M/Z"].astype(str), sample["Intensity"]))
      row["mouse_id"] = group_id
      row["HasCancer"] = label
      rows.append(row)

df = pd.DataFrame(rows)
df["HasCancer"] = df["HasCancer"].astype("category")
df["mouse_id"] = df["mouse_id"].astype("category")
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For Grouped Non-IID data
split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
    random_state=split_random_state,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()